# WAXAL-NET Fine-tuning: Dagbani
Fine-tunes Whisper Tiny, Whisper Small, and MMS-300M on WAXAL Dagbani data.
Runs evaluation on the WAXAL test set.

In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "--quiet",
    "torch==2.8.0", "torchaudio==2.8.0",
    "accelerate==1.10.1", "transformers==4.51.3",
    "huggingface_hub==0.32.6", "datasets[audio]==3.6.0",
    "numpy==1.26.4", "jiwer==4.0.0", "omnilingual-asr==0.1.0",
    "PyYAML==6.0.3", "soundfile==0.13.1", "wandb==0.19.11"
], check=True)
print("Dependencies installed.")

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache"
os.environ["WANDB_DISABLED"] = "true"
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.makedirs("/kaggle/working/outputs", exist_ok=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
print("Environment configured.")

In [ ]:
import sys
sys.path.insert(0, "/kaggle/input/waxal-asr-code/asr_waxal")
print("Repository path added.")

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "src.train",
     "--config", "configs/experiment/finetune_dagbani_whisper_tiny.yaml"],
    cwd="/kaggle/input/waxal-asr-code/asr_waxal",
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Training failed for dagbani whisper_tiny")

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "src.evaluate",
     "--config", "configs/experiment/finetune_dagbani_whisper_tiny.yaml",
     "--checkpoint", "/kaggle/working/checkpoints/finetune_dagbani_whisper_tiny/best_model"],
    cwd="/kaggle/input/waxal-asr-code/asr_waxal",
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Evaluation failed for dagbani whisper_tiny")

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "src.train",
     "--config", "configs/experiment/finetune_dagbani_whisper_small.yaml"],
    cwd="/kaggle/input/waxal-asr-code/asr_waxal",
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Training failed for dagbani whisper_small")

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "src.evaluate",
     "--config", "configs/experiment/finetune_dagbani_whisper_small.yaml",
     "--checkpoint", "/kaggle/working/checkpoints/finetune_dagbani_whisper_small/best_model"],
    cwd="/kaggle/input/waxal-asr-code/asr_waxal",
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Evaluation failed for dagbani whisper_small")

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "src.train",
     "--config", "configs/experiment/finetune_dagbani_mms_300m.yaml"],
    cwd="/kaggle/input/waxal-asr-code/asr_waxal",
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Training failed for dagbani mms_300m")

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "src.evaluate",
     "--config", "configs/experiment/finetune_dagbani_mms_300m.yaml",
     "--checkpoint", "/kaggle/working/checkpoints/finetune_dagbani_mms_300m/best_model"],
    cwd="/kaggle/input/waxal-asr-code/asr_waxal",
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Evaluation failed for dagbani mms_300m")

In [ ]:
import json
from pathlib import Path

models = ["whisper_tiny", "whisper_small", "mms_300m"]
print(f"=== Results for Dagbani ===")
for model in models:
    p = Path(f"/kaggle/working/outputs/finetune_dagbani_{model}/metrics.json")
    if p.exists():
        m = json.loads(p.read_text())
        if m.get("status") == "N/A":
            print(f"  {model}: N/A")
        else:
            print(f"  {model}: WER={m['wer_percent']:.1f}% CER={m['cer_percent']:.1f}%")
    else:
        print(f"  {model}: not found")